## Modeling melt transport
The governing equations of conservation of mass, momentum, and energy are modified to account for the melt generation and transport (described in section 2.1.1 in Dannberg and Heister, 2016).

In ASPECT, melt fraction is tracked using a compositional field named `porosity`. We will describe the .prm file of the accompanying model in the subsequent sections.

The initial setup is shown below:

<div>
<img src='./images/model_setup.png' width="500"/>
<figcaption align = "left"> Initial model setup showing the peridotite composition (scale bar) along with the velocity field (arows). </figcaption>
</div>

### Geometry Model
We use a rectangular box of 2000x1000 km, and 2 repetitions along the X direction to maintain a cell aspect ratio of 1.

```
subsection Geometry model
  set Model name = box
    subsection Box
      set X extent = 2000000
      set Y extent = 1000000
      set X repetitions = 2
   end
end
```

### Initial Compositional Field

We use a circular region of radius 100 km at the center of the box to initialize the peridotite compositional field, which will be used to track how much of the material has been molten. The porosity field (melt) is initialized to zero everywhere.

```
subsection Compositional fields
   set Number of fields = 2
   set Names of fields  = peridotite, porosity
end

subsection Initial composition model
  set Model name = function

  subsection Function
    set Function constants        = r=100000, amplitude=1
    set Function expression       = if((x-1000e3)*(x-1000e3)+y*y<r*r,amplitude,0);0
  end
end
```



### Initial Temperature Field
The peridotite composition is initialized with a temperature field with higher temperature (100 K above the background) at the center of the box. The rest of the model is initialized with an adiabatic temperature profile.

```
subsection Initial temperature model
  set List of model names =  adiabatic, function
  set List of model operators = add
  
  subsection Adiabatic
    set Age top boundary layer    = 1e8
    set Age bottom boundary layer = 5e7
  end

  subsection Function
    set Function constants        = r=100000, amplitude=100
    set Function expression       = if((x-1000e3)*(x-1000e3)+y*y<r*r,amplitude,0)
  end
end
```

### Material Model
We use `melt simple` model that follows porosity according to the melting model for dry peridotite of Katz et al., 2003.

```
subsection Material model
  set Model name = melt simple

  subsection Melt simple
    set Reference shear viscosity = 2e20
    set Reference solid density = 3300
    set Reference melt density = 3200
    set Melting time scale for operator splitting = 1e4
    set Melt extraction depth = 10e3
    set Exponential melt weakening factor = 20
  end
end
```

The dependence of porosity on viscosity is governed by `Exponential melt weakening factor`. Melt is assumed to be $100 kg/m^3$ less dense than the solid.

The parameter, `Melting time scale for operator splitting` defines the time after which the deviation of the porosity from the equilibrium melt fraction will be reduced to a fraction of $1/e$.

### Melt Settings
The following parameters solve the McKenzie Equations to track the flow of melt.

```
subsection Melt settings
  set Include melt transport = true
  set Heat advection by melt = true
end
```

#### References

- Katz, Richard F., Marc Spiegelman, and Charles H. Langmuir. A new parameterization of hydrous mantle melting. Geochemistry, Geophysics, Geosystems 4.9 (2003).
- McKenzie, D. A. N. The generation and compaction of partially molten rock."Journal of petrology 25.3 (1984): 713-765.

 &nbsp;<div style="text-align: right">  
    &rarr; <b>NEXT: [Plotting melt model](./6_plotting_melt_model.ipynb) </b> <a href=""></a> &nbsp;&nbsp;
     <img src="../assets/education-gem-notebooks_icon.png" alt="icon"  style="width:4%">
  </div>